In [1]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 KB 2.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 2.0 MB/s eta 0:00:0000:0100:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 2.3 MB/s eta 0:00:0000:0100:01


In [5]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke("Top new in the AI")

In [4]:
pip install -U ddgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 KB 601.7 kB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [8]:
print( "1" , search_tool.name)
print("2" , search_tool.description)
print("3" ,search_tool.args)

1 duckduckgo_search
2 A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
3 {'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


In [9]:
# Built in Shell Tool

In [10]:
from langchain_community.tools import ShellTool

In [11]:
shell_tool = ShellTool()

In [12]:
results = shell_tool.invoke("ls")

Executing command:
 ls


/mnt/c/Users/vinay kumar/Desktop/LangChain_and_RAG/venv/lib/python3.10/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


In [13]:
print(results)

langchain_tools.ipynb



# Custom Tools

In [14]:
from langchain_core.tools import tool 

In [ ]:
def multiply(a,b):
  """Multiply two number"""
  return a*b

In [15]:
# step 2 - Add type hints 
def multiply(a: int, b: int) -> int:
  """Multiply two numbers"""
  return a*b

In [16]:
@tool
def multiply(a: int, b: int) -> int:
  """Multiply two numbers"""
  return a*b

In [17]:
results = multiply.invoke({"a":3,"b":5})

In [18]:
print(results)

15


In [19]:
print(multiply)
print(multiply.description)
print(multiply.args)

name='multiply' description='Multiply two numbers' args_schema=<class 'langchain_core.utils.pydantic.multiply'> func=<function multiply at 0x712fec2ede10>
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [20]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


# Method 2 - Using Structured Tool 

In [23]:
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool



In [24]:
class MultiplyInput(BaseModel):
  a: int = Field(required=True, description="First number to add")
  b: int = Field(required=True,description="The second number to add" )

/tmp/ipykernel_3606/656055119.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="First number to add")
/tmp/ipykernel_3606/656055119.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True,description="The second number to add" )


In [25]:
def multiply_func(a : int, b : int) -> int:
  return a*b

In [27]:
multiply_tool = StructuredTool.from_function(
  func=multiply_func,
  name="multiply",
  description="Multiply two number",
  args_schema=MultiplyInput
)

In [29]:
result = multiply_tool.invoke({'a':3, 'b':4})

print(result)
print(multiply_tool.description)
print(multiply_tool.name)
print(multiply_tool.args)


12
Multiply two number
multiply
{'a': {'description': 'First number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


# Method 3 - Using BaseTool Class

In [32]:
from langchain.tools import BaseTool
from typing import Type

In [33]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_3606/908171234.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_3606/908171234.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(required=True, description="The second number to add")


In [34]:
class MultiplyTool(BaseTool):
  name: str = "Multiply"
  description: str = "Multiply two number"

  args_schema: Type[BaseModel] = MultiplyInput
  def _run(slef, a: int, b: int) -> int:
    return a*b

In [35]:
multiply_tool = MultiplyTool()

In [36]:
result = multiply_tool.invoke({'a':3,'b':3})

In [37]:
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
Multiply
Multiply two number
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


In [39]:
from langchain_core.tools import tool

#Custom 
@tool
def add(a: int, b: int)->int:
  """Add two number"""
  return a+b


@tool
def multiply(a:int,b: int) -> int:
  """Multiply two numbers"""
  return a*b



In [43]:
class MathToolkit:
  def get_tools(self):
    return [add,multiply]

In [44]:
tool_kit = MathToolkit()
tools = tool_kit.get_tools()

In [47]:
for tool in tools:
  print(tool.name , "=>" , tool.description)

add => Add two number
multiply => Multiply two numbers
